# 4-1 素のPythonで3パターン認識
`|`、`-`、`/` の16×16パターンを、Pythonのリストとfor文で学習します。


In [ ]:
# Colab単独で実行できるよう、15件の16×16データをNotebook内に持たせています。
PATTERNS = {'0_01.txt': ['                ', '                ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '                ', '                '], '0_02.txt': ['                ', '                ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '        ■       ', '                ', '                '], '0_03.txt': ['                ', '                ', '                ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '      ■         ', '                ', '                ', '                '], '0_04.txt': ['                ', '                ', '                ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '         ■      ', '                ', '                ', '                '], '0_05.txt': ['                ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '       ■        ', '                '], '1_01.txt': ['                ', '                ', '                ', '                ', '                ', '                ', '                ', '  ■■■■■■■■■■■■  ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '                '], '1_02.txt': ['                ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '  ■■■■■■■■■■■■  ', '                ', '                ', '                ', '                ', '                ', '                ', '                '], '1_03.txt': ['                ', '                ', '                ', '                ', '                ', '                ', '   ■■■■■■■■■■   ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '                '], '1_04.txt': ['                ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '   ■■■■■■■■■■   ', '                ', '                ', '                ', '                ', '                ', '                '], '1_05.txt': ['                ', '                ', '                ', '                ', '                ', '                ', '                ', ' ■■■■■■■■■■■■■■ ', '                ', '                ', '                ', '                ', '                ', '                ', '                ', '                '], '2_01.txt': ['                ', '                ', '             ■  ', '            ■   ', '           ■    ', '          ■     ', '         ■      ', '        ■       ', '       ■        ', '      ■         ', '     ■          ', '    ■           ', '   ■            ', '  ■             ', '                ', '                '], '2_02.txt': ['                ', '                ', '            ■   ', '           ■    ', '          ■     ', '         ■      ', '        ■       ', '       ■        ', '      ■         ', '     ■          ', '    ■           ', '   ■            ', '  ■             ', '                ', '                ', '                '], '2_03.txt': ['                ', '              ■ ', '             ■  ', '            ■   ', '           ■    ', '          ■     ', '         ■      ', '        ■       ', '       ■        ', '      ■         ', '     ■          ', '    ■           ', '   ■            ', '  ■             ', '                ', '                '], '2_04.txt': ['                ', '                ', '                ', '             ■  ', '            ■   ', '           ■    ', '          ■     ', '         ■      ', '        ■       ', '       ■        ', '      ■         ', '     ■          ', '    ■           ', '   ■            ', '                ', '                '], '2_05.txt': ['                ', '                ', '              ■ ', '             ■  ', '            ■   ', '           ■    ', '          ■     ', '         ■      ', '        ■       ', '       ■        ', '      ■         ', '     ■          ', '    ■           ', '   ■            ', '                ', '                ']}

LABELS = {0: "|", 1: "-", 2: "/"}

def bits_from_lines(lines):
    return [1.0 if ch == "■" else 0.0 for line in lines for ch in line]

training_data = []
for filename, lines in sorted(PATTERNS.items()):
    training_data.append({
        "filename": filename,
        "class_id": int(filename[0]),
        "bits": bits_from_lines(lines)
    })

print("学習データ数 =", len(training_data))


In [ ]:
# 3種類の代表データを表示
for filename in ["0_01.txt", "1_01.txt", "2_01.txt"]:
    class_id = int(filename[0])
    print(f"{LABELS[class_id]}  ({filename})")
    print("\n".join(PATTERNS[filename]))
    print()


In [ ]:
import random

random.seed(0)

weights = [[0.0] * 256 for _ in range(3)]
bias = [0.0, 0.0, 0.0]

rate = 0.04 / 15
epochs = 600

for epoch in range(epochs):
    loss = 0.0
    grad_w = [[0.0] * 256 for _ in range(3)]
    grad_b = [0.0, 0.0, 0.0]

    for td in training_data:
        bits = td["bits"]
        target = [0.0, 0.0, 0.0]
        target[td["class_id"]] = 1.0

        prediction = []
        for cls in range(3):
            value = bias[cls]
            for i in range(256):
                value += bits[i] * weights[cls][i]
            prediction.append(value)

        for cls in range(3):
            diff = prediction[cls] - target[cls]
            loss += diff ** 2
            gradient = 2 * diff
            grad_b[cls] += gradient
            for i in range(256):
                grad_w[cls][i] += gradient * bits[i]

    for cls in range(3):
        bias[cls] -= rate * grad_b[cls]
        for i in range(256):
            weights[cls][i] -= rate * grad_w[cls][i]

    if epoch % 100 == 0:
        print("epoch =", epoch, "loss =", round(loss, 6))

print("\n----- result -----")
for td in training_data:
    prediction = []
    for cls in range(3):
        value = bias[cls]
        for i in range(256):
            value += td["bits"][i] * weights[cls][i]
        prediction.append(value)
    answer = max(range(3), key=lambda c: prediction[c])
    print(td["filename"], "target =", LABELS[td["class_id"]],
          "answer =", LABELS[answer],
          "prediction =", [round(v, 3) for v in prediction])
